# Notebook 1 v3.3 — Downloaded Artifact Pool Audit + Downloader-aware URL Worklist

This notebook audits downloaded artifacts, merges `dataset_structure.json` signals, and creates URL worklists using the same rule as the artifact downloader:

```python
if item.get("accessible") is True:
    url = item.get("redirected_url") or item.get("url")
```

Outputs:
- `downloaded_artifact_pool_audit_v3_3_schema.csv`
- `downloaded_artifact_pool_audit_v3_3_schema.xlsx`
- `rerun_ready_v1_url_worklist.csv`
- `manual_check_pool_v1_url_worklist.csv`
- `high_priority_70_url_worklist.csv`

In [1]:
from pathlib import Path
import json
from collections import Counter
from urllib.parse import urlparse
import pandas as pd

DATA_ROOT = Path("/mydata/doc2validate/data")
DOWNLOADED_ARTIFACTS_DIR = DATA_ROOT / "downloaded_artifacts"
STRUCTURED_DOCS_DIR = DATA_ROOT / "structured_docs"
RUN_ROOT = Path("/mydata/doc2validate/results/runs/scidata_4293")
ANALYSIS_DIR = RUN_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = ANALYSIS_DIR / "downloaded_artifact_pool_audit_v3_3_schema.csv"
OUTPUT_XLSX = ANALYSIS_DIR / "downloaded_artifact_pool_audit_v3_3_schema.xlsx"

print("Downloaded artifacts dir:", DOWNLOADED_ARTIFACTS_DIR)
print("Structured docs dir:", STRUCTURED_DOCS_DIR)
print("Downloaded artifacts exists:", DOWNLOADED_ARTIFACTS_DIR.exists())
print("Structured docs exists:", STRUCTURED_DOCS_DIR.exists())

Downloaded artifacts dir: /mydata/doc2validate/data/downloaded_artifacts
Structured docs dir: /mydata/doc2validate/data/structured_docs
Downloaded artifacts exists: True
Structured docs exists: True


In [2]:
MANIFEST_NAME = "ARTIFACT_DOWNLOAD_MANIFEST.json"
TABULAR_SUFFIXES = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet", ".json", ".jsonl"}
STRICT_TABULAR_SUFFIXES = {".csv", ".tsv", ".xlsx", ".xls", ".parquet"}
ARCHIVE_SUFFIXES = {".zip", ".tar", ".gz", ".tgz", ".tar.gz", ".7z", ".rar", ".bz2", ".xz"}
KNOWN_PROVIDER_DIRS = {"github", "zenodo", "direct", "figshare", "osf", "generic", "dataverse"}
TABULAR_FORMATS = {"csv", "tsv", "xlsx", "xls", "excel", "parquet", "json", "jsonl", "txt"}
STRICT_TABULAR_FORMATS = {"csv", "tsv", "xlsx", "xls", "excel", "parquet"}
PRIMARY_ROLE_KEYWORDS = {"primary", "primary_data", "main", "main_data", "raw", "raw_data"}
DERIVED_ROLE_KEYWORDS = {"derived", "derived_data", "processed", "intermediate"}
METADATA_ROLE_KEYWORDS = {"metadata", "data_dictionary", "codebook", "documentation"}
ANNOTATION_ROLE_KEYWORDS = {"annotation", "annotations", "label", "labels"}
SOFTWARE_ROLE_KEYWORDS = {"software", "code", "script", "runtime", "environment", "notebook"}
NON_TABULAR_OR_ENV_FORMATS = {"binary", "image", "images", "hdf5", "h5", "netcdf", "nc", "database", "mongodb", "sql", "sqlite", "vasp", "mat", "matlab", "rdata", "rds"}

In [3]:
def safe_load_json(path: Path):
    if not path or not Path(path).exists(): return None
    for enc in ["utf-8", "utf-8-sig"]:
        try:
            with open(path, "r", encoding=enc) as f: return json.load(f)
        except Exception: pass
    try: return {"_load_error": Path(path).read_text(errors="replace")[:500]}
    except Exception as e: return {"_load_error": str(e)}

def load_json(path: Path):
    data = safe_load_json(path)
    return data if isinstance(data, dict) else {}

def is_hidden_or_system(path: Path) -> bool:
    return path.name.startswith(".") or path.name in {"__MACOSX", ".DS_Store", "__pycache__"}

def suffix_of(path: Path) -> str:
    name = path.name.lower()
    if name.endswith(".tar.gz"): return ".tar.gz"
    return path.suffix.lower()

def is_archive(path: Path) -> bool: return suffix_of(path) in ARCHIVE_SUFFIXES

def is_tabular_like(path: Path) -> bool: return suffix_of(path) in TABULAR_SUFFIXES

def is_strict_tabular(path: Path) -> bool: return suffix_of(path) in STRICT_TABULAR_SUFFIXES

def iter_real_files(article_dir: Path):
    for p in article_dir.rglob("*"):
        if is_hidden_or_system(p): continue
        if p.is_file() and p.name != MANIFEST_NAME: yield p

def top_level_entries_except_manifest(article_dir: Path):
    return [p for p in article_dir.iterdir() if not is_hidden_or_system(p) and p.name != MANIFEST_NAME]

def is_under_extracted(article_dir: Path, path: Path) -> bool: return "extracted" in path.relative_to(article_dir).parts

def is_under_archive_dir(article_dir: Path, path: Path) -> bool: return "archive" in path.relative_to(article_dir).parts

def compact_dict(d): return json.dumps(d, ensure_ascii=False, sort_keys=True) if d else ""

def compact_list(xs, max_items=20):
    if not xs: return ""
    xs = [str(x) for x in xs if str(x) != ""]
    if not xs: return ""
    if len(xs) > max_items: return "; ".join(xs[:max_items]) + f"; ... (+{len(xs)-max_items})"
    return "; ".join(xs)

def normalize_str(x): return "" if x is None else (x.strip() if isinstance(x, str) else str(x).strip())
def normalize_lower(x): return normalize_str(x).lower().strip()

In [4]:
def find_dataset_structure_path(article_id: str):
    candidates = [STRUCTURED_DOCS_DIR / article_id / "dataset_structure.json", DATA_ROOT / "dataset_structure" / article_id / "dataset_structure.json"]
    return next((p for p in candidates if p.exists()), None)

def unwrap_schema(schema):
    if not isinstance(schema, dict): return schema
    for key in ["result", "dataset_structure", "structure", "logical_schema", "schema"]:
        if isinstance(schema.get(key), dict): return schema[key]
    return schema

def get_files_from_schema(schema):
    schema = unwrap_schema(schema)
    if not isinstance(schema, dict): return []
    for key in ["files", "dataset_files", "logical_files", "data_files", "artifacts"]:
        if isinstance(schema.get(key), list): return schema[key]
    org = schema.get("organization")
    if isinstance(org, dict):
        for key in ["files", "data_files", "logical_files", "artifacts"]:
            if isinstance(org.get(key), list): return org[key]
    return []

def get_validation_targets(schema):
    schema = unwrap_schema(schema)
    if not isinstance(schema, dict): return []
    val = schema.get("validation_targets", [])
    if isinstance(val, list): return val
    return [val] if val else []

def get_execution_notes(schema):
    schema = unwrap_schema(schema)
    if not isinstance(schema, dict): return []
    notes=[]
    for key in ["execution_relevant_notes", "execution_notes", "runtime_notes", "validation_notes"]:
        val=schema.get(key)
        if isinstance(val, list): notes.extend([normalize_str(v) for v in val if normalize_str(v)])
        elif isinstance(val, str) and val.strip(): notes.append(val.strip())
    return notes

def get_context_summary(schema):
    schema = unwrap_schema(schema)
    if not isinstance(schema, dict): return ""
    val=schema.get("context_summary", "")
    if isinstance(val, str): return val
    if isinstance(val, dict): return json.dumps(val, ensure_ascii=False)[:1000]
    return normalize_str(val)

In [5]:
def file_role(f):
    if not isinstance(f, dict): return ""
    for key in ["role", "file_role", "schema_type", "category"]:
        val = normalize_lower(f.get(key))
        if val: return val
    return ""

def file_format(f):
    if not isinstance(f, dict): return ""
    for key in ["format", "expected_format", "file_format"]:
        val=normalize_lower(f.get(key))
        if val: return val.replace(".", "")
    return ""

def file_name_or_path(f):
    if not isinstance(f, dict): return ""
    vals=[]
    for key in ["logical_name", "name", "file_name", "filename", "path", "relative_path", "file_pattern", "expected_path", "expected_file", "location"]:
        val=normalize_str(f.get(key))
        if val: vals.append(val)
    return " | ".join(vals)

def file_has_path_or_pattern(f):
    if not isinstance(f, dict): return False
    for key in ["path", "relative_path", "file_pattern", "expected_path", "expected_file", "location", "directory"]:
        if normalize_str(f.get(key)): return True
    return False

def normalize_column_dict(columns_dict):
    out=[]
    for name, meta in columns_dict.items():
        if isinstance(meta, dict):
            item={"name": name}; item.update(meta); out.append(item)
        else:
            out.append({"name": name, "description": normalize_str(meta)})
    return out

def file_columns(f):
    if not isinstance(f, dict): return []
    structure = f.get("structure")
    if isinstance(structure, dict):
        val = structure.get("columns")
        if isinstance(val, dict): return normalize_column_dict(val)
        if isinstance(val, list): return val
        for subkey in ["fields", "variables"]:
            subval=structure.get(subkey)
            if isinstance(subval, dict): return normalize_column_dict(subval)
            if isinstance(subval, list): return subval
    for key in ["columns", "expected_columns", "column_semantics", "fields", "variables"]:
        val=f.get(key)
        if isinstance(val, dict): return normalize_column_dict(val)
        if isinstance(val, list): return val
    schema=f.get("schema")
    if isinstance(schema, dict):
        for subkey in ["columns", "fields", "variables"]:
            subval=schema.get(subkey)
            if isinstance(subval, dict): return normalize_column_dict(subval)
            if isinstance(subval, list): return subval
    return []

def column_has_semantics(col):
    if not isinstance(col, dict): return False
    semantic_keys=["semantic", "semantics", "semantic_role", "semantic_type", "description", "meaning", "definition", "role", "unit", "units", "data_type", "datatype", "type", "expected_values", "allowed_values", "notes"]
    return any(normalize_str(col.get(k)) for k in semantic_keys)

def file_has_column_semantics(f):
    cols=file_columns(f)
    return bool(cols) and any(column_has_semantics(c) for c in cols)

def file_column_count(f): return len(file_columns(f))
def is_primary_role(role): return any(k in normalize_lower(role) for k in PRIMARY_ROLE_KEYWORDS)
def is_derived_role(role): return any(k in normalize_lower(role) for k in DERIVED_ROLE_KEYWORDS)
def is_metadata_role(role): return any(k in normalize_lower(role) for k in METADATA_ROLE_KEYWORDS)
def is_annotation_role(role): return any(k in normalize_lower(role) for k in ANNOTATION_ROLE_KEYWORDS)
def is_software_role(role): return any(k in normalize_lower(role) for k in SOFTWARE_ROLE_KEYWORDS)
def is_tabular_format(fmt): return normalize_lower(fmt).replace(".", "") in TABULAR_FORMATS
def is_strict_tabular_format(fmt): return normalize_lower(fmt).replace(".", "") in STRICT_TABULAR_FORMATS
def is_non_tabular_or_env_format(fmt): return normalize_lower(fmt).replace(".", "") in NON_TABULAR_OR_ENV_FORMATS

In [6]:
def extract_schema_features(article_id: str):
    path = find_dataset_structure_path(article_id)
    if path is None: return {"has_dataset_structure": False, "dataset_structure_path": ""}
    raw_schema = safe_load_json(path)
    if not isinstance(raw_schema, dict): return {"has_dataset_structure": False, "dataset_structure_path": str(path), "dataset_structure_load_error": "not_a_dict_or_load_failed"}
    schema = unwrap_schema(raw_schema)
    if not isinstance(schema, dict): return {"has_dataset_structure": False, "dataset_structure_path": str(path), "dataset_structure_load_error": "unwrapped_schema_not_dict"}
    files = get_files_from_schema(raw_schema)
    validation_targets = get_validation_targets(raw_schema)
    execution_notes = get_execution_notes(raw_schema)
    roles=[file_role(f) for f in files if isinstance(f, dict)]
    formats=[file_format(f) for f in files if isinstance(f, dict)]
    formats=[f for f in formats if f]
    primary_files=[f for f in files if isinstance(f, dict) and is_primary_role(file_role(f))]
    derived_files=[f for f in files if isinstance(f, dict) and is_derived_role(file_role(f))]
    metadata_files=[f for f in files if isinstance(f, dict) and is_metadata_role(file_role(f))]
    annotation_files=[f for f in files if isinstance(f, dict) and is_annotation_role(file_role(f))]
    software_files=[f for f in files if isinstance(f, dict) and is_software_role(file_role(f))]
    tabular_files=[f for f in files if isinstance(f, dict) and is_tabular_format(file_format(f))]
    strict_tabular_files=[f for f in files if isinstance(f, dict) and is_strict_tabular_format(file_format(f))]
    primary_tabular_files=[f for f in primary_files if is_tabular_format(file_format(f))]
    primary_strict_tabular_files=[f for f in primary_files if is_strict_tabular_format(file_format(f))]
    non_tabular_or_env_files=[f for f in files if isinstance(f, dict) and is_non_tabular_or_env_format(file_format(f))]
    files_with_columns=[f for f in files if isinstance(f, dict) and len(file_columns(f)) > 0]
    files_with_column_semantics=[f for f in files if isinstance(f, dict) and file_has_column_semantics(f)]
    known_path_files=[f for f in files if isinstance(f, dict) and file_has_path_or_pattern(f)]
    organization=schema.get("organization", "")
    organization_type=normalize_str(organization.get("type") or organization.get("organization_type") or organization.get("structure") or organization.get("description")) if isinstance(organization, dict) else normalize_str(organization)
    role_counts=Counter([r for r in roles if r]); format_counts=Counter([f for f in formats if f])
    primary_formats=[file_format(f) for f in primary_files if file_format(f)]
    return {
        "has_dataset_structure": True, "dataset_structure_path": str(path), "organization_type": organization_type, "structure_confidence": schema.get("structure_confidence", schema.get("confidence", "")),
        "logical_file_count": len(files), "primary_file_count": len(primary_files), "derived_file_count": len(derived_files), "metadata_file_count": len(metadata_files), "annotation_file_count": len(annotation_files), "software_file_count": len(software_files),
        "tabular_logical_file_count": len(tabular_files), "strict_tabular_logical_file_count": len(strict_tabular_files), "primary_tabular_file_count": len(primary_tabular_files), "primary_strict_tabular_file_count": len(primary_strict_tabular_files), "non_tabular_or_env_logical_file_count": len(non_tabular_or_env_files),
        "files_with_columns_count": len(files_with_columns), "files_with_column_semantics_count": len(files_with_column_semantics), "total_declared_column_count": sum(file_column_count(f) for f in files if isinstance(f, dict)), "known_path_or_pattern_count": len(known_path_files),
        "has_primary_tabular": len(primary_tabular_files)>0, "has_primary_strict_tabular": len(primary_strict_tabular_files)>0, "has_column_semantics": len(files_with_column_semantics)>0, "has_validation_targets": len(validation_targets)>0, "has_execution_relevant_notes": len(execution_notes)>0,
        "logical_formats": compact_list(sorted(set(formats))), "primary_formats": compact_list(sorted(set(primary_formats))), "role_counts": compact_dict(dict(role_counts)), "format_counts": compact_dict(dict(format_counts)),
        "validation_target_count": len(validation_targets), "execution_relevant_note_count": len(execution_notes), "context_summary_preview": get_context_summary(raw_schema)[:500],
        "sample_logical_files": compact_list([file_name_or_path(f) for f in files[:20] if isinstance(f, dict)], max_items=20),
        "sample_primary_files": compact_list([file_name_or_path(f) for f in primary_files[:20] if isinstance(f, dict)], max_items=20),
        "sample_files_with_column_semantics": compact_list([file_name_or_path(f) for f in files_with_column_semantics[:20] if isinstance(f, dict)], max_items=20),
    }

In [7]:
# Quick sanity check on examples
for article_id in ["s41597-024-04232-w", "s41597-023-02060-y", "s41597-020-0407-9"]:
    feats = extract_schema_features(article_id)
    print("\n" + "="*80)
    print(article_id)
    for k in ["logical_file_count", "primary_file_count", "tabular_logical_file_count", "primary_tabular_file_count", "files_with_column_semantics_count", "known_path_or_pattern_count", "logical_formats", "primary_formats", "role_counts"]:
        print(k, "=", feats.get(k))


s41597-024-04232-w
logical_file_count = 8
primary_file_count = 4
tabular_logical_file_count = 8
primary_tabular_file_count = 4
files_with_column_semantics_count = 8
known_path_or_pattern_count = 8
logical_formats = csv; tsv
primary_formats = csv; tsv
role_counts = {"derived_data": 3, "metadata": 1, "primary_data": 4}

s41597-023-02060-y
logical_file_count = 8
primary_file_count = 8
tabular_logical_file_count = 8
primary_tabular_file_count = 8
files_with_column_semantics_count = 8
known_path_or_pattern_count = 8
logical_formats = csv
primary_formats = csv
role_counts = {"primary_data": 8}

s41597-020-0407-9
logical_file_count = 5
primary_file_count = 2
tabular_logical_file_count = 2
primary_tabular_file_count = 0
files_with_column_semantics_count = 2
known_path_or_pattern_count = 5
logical_formats = binary; json; text; unknown
primary_formats = binary; unknown
role_counts = {"derived_data": 1, "metadata": 2, "primary_data": 2}


In [8]:
article_dirs = sorted([p for p in DOWNLOADED_ARTIFACTS_DIR.iterdir() if p.is_dir()])
print("Article dirs:", len(article_dirs))
rows=[]
for article_dir in article_dirs:
    article_id = article_dir.name
    top_entries=top_level_entries_except_manifest(article_dir)
    top_dirs=[p.name for p in top_entries if p.is_dir()]
    top_files=[p.name for p in top_entries if p.is_file()]
    real_files=list(iter_real_files(article_dir))
    real_dirs=[p for p in article_dir.rglob("*") if p.is_dir() and not is_hidden_or_system(p)]
    archive_files=[p for p in real_files if is_archive(p)]
    archive_dir_files=[p for p in real_files if is_under_archive_dir(article_dir, p)]
    extracted_files=[p for p in real_files if is_under_extracted(article_dir, p)]
    non_archive_non_manifest_files=[p for p in real_files if not is_under_archive_dir(article_dir, p)]
    extracted_tabular_files=[p for p in extracted_files if is_tabular_like(p)]
    extracted_strict_tabular_files=[p for p in extracted_files if is_strict_tabular(p)]
    non_archive_tabular_files=[p for p in non_archive_non_manifest_files if is_tabular_like(p)]
    non_archive_strict_tabular_files=[p for p in non_archive_non_manifest_files if is_strict_tabular(p)]
    file_suffix_counts=Counter(suffix_of(p) or "[no_suffix]" for p in real_files)
    extracted_suffix_counts=Counter(suffix_of(p) or "[no_suffix]" for p in extracted_files)
    non_archive_suffix_counts=Counter(suffix_of(p) or "[no_suffix]" for p in non_archive_non_manifest_files)
    provider_dirs_present=sorted([d for d in top_dirs if d.lower() in KNOWN_PROVIDER_DIRS])
    if len(top_entries)==0: artifact_presence="manifest_only"
    elif len(real_files)==0: artifact_presence="dirs_but_no_files"
    else: artifact_presence="has_real_artifact"
    has_extracted_content=len(extracted_files)>0
    has_non_archive_content=len(non_archive_non_manifest_files)>0
    has_archive_backup=len(archive_files)>0 or len(archive_dir_files)>0
    if artifact_presence != "has_real_artifact": downloaded_pool_status_v2="exclude_no_downloaded_content"
    elif len(extracted_strict_tabular_files)>0: downloaded_pool_status_v2="has_extracted_strict_tabular_candidate"
    elif len(extracted_tabular_files)>0: downloaded_pool_status_v2="has_extracted_tabular_like_candidate"
    elif len(non_archive_strict_tabular_files)>0: downloaded_pool_status_v2="has_non_archive_strict_tabular_candidate"
    elif len(non_archive_tabular_files)>0: downloaded_pool_status_v2="has_non_archive_tabular_like_candidate"
    elif has_extracted_content: downloaded_pool_status_v2="has_extracted_non_tabular_content"
    elif has_archive_backup and not has_non_archive_content: downloaded_pool_status_v2="archive_backup_only_no_extracted_content"
    else: downloaded_pool_status_v2="has_non_archive_non_tabular_content"
    row={
        "article_id": article_id, "article_dir": str(article_dir), "artifact_presence": artifact_presence, "downloaded_pool_status_v2": downloaded_pool_status_v2,
        "top_level_dirs_except_manifest": compact_list(top_dirs), "top_level_files_except_manifest": compact_list(top_files), "provider_dirs_present": compact_list(provider_dirs_present),
        "has_archive_backup": has_archive_backup, "has_extracted_content": has_extracted_content, "has_non_archive_content": has_non_archive_content,
        "real_file_count_except_manifest": len(real_files), "real_dir_count": len(real_dirs), "archive_file_count": len(archive_files), "archive_dir_file_count": len(archive_dir_files),
        "extracted_file_count": len(extracted_files), "non_archive_file_count": len(non_archive_non_manifest_files),
        "extracted_tabular_like_file_count": len(extracted_tabular_files), "extracted_strict_tabular_file_count": len(extracted_strict_tabular_files),
        "non_archive_tabular_like_file_count": len(non_archive_tabular_files), "non_archive_strict_tabular_file_count": len(non_archive_strict_tabular_files),
        "file_suffix_counts_all": compact_dict(dict(file_suffix_counts)), "extracted_suffix_counts": compact_dict(dict(extracted_suffix_counts)), "non_archive_suffix_counts": compact_dict(dict(non_archive_suffix_counts)),
        "sample_extracted_files": compact_list([str(p.relative_to(article_dir)) for p in extracted_files[:30]], max_items=30),
        "sample_extracted_tabular_files": compact_list([str(p.relative_to(article_dir)) for p in extracted_tabular_files[:20]], max_items=20),
    }
    row.update(extract_schema_features(article_id))
    rows.append(row)
df = pd.DataFrame(rows)
df.head()

Article dirs: 187


,article_id,article_dir,artifact_presence,downloaded_pool_status_v2,top_level_dirs_except_manifest,top_level_files_except_manifest,provider_dirs_present,has_archive_backup,has_extracted_content,has_non_archive_content,...,logical_formats,primary_formats,role_counts,format_counts,validation_target_count,execution_relevant_note_count,context_summary_preview,sample_logical_files,sample_primary_files,sample_files_with_column_semantics
0,s41597-019-0021-x,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_non_archive_strict_tabular_candidate,github,,github,False,False,True,...,,,,,0,0,"{""requested_strategy"": ""section_focused_contex...",,,
1,s41597-019-0035-4,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_extracted_strict_tabular_candidate,github,,github,True,True,True,...,csv; json,json,"{""annotation"": 1, ""derived_data"": 3, ""metadata...","{""csv"": 4, ""json"": 2}",4,6,"{""requested_strategy"": ""section_focused_contex...",anatomical_iqms | https://figshare.com/article...,api_data_stream | https://mriqc.nimh.nih.gov/a...,anatomical_iqms | https://figshare.com/article...
2,s41597-019-0098-2,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_extracted_strict_tabular_candidate,github,,github,True,True,True,...,binary; text; tsv; unknown,binary,"{""annotation"": 1, ""derived_data"": 4, ""metadata...","{""binary"": 4, ""text"": 1, ""tsv"": 1, ""unknown"": 2}",4,6,"{""requested_strategy"": ""section_focused_contex...",raw_data_tar_gz | ftp://ftp-trace.ncbi.nlm.nih...,raw_data_tar_gz | ftp://ftp-trace.ncbi.nlm.nih...,sequence_index_file | https://github.com/genom...
3,s41597-019-0213-4,/mydata/doc2validate/data/downloaded_artifacts...,manifest_only,exclude_no_downloaded_content,,,,False,False,False,...,csv; json; php; sql; unknown,csv; unknown,"{""derived_data"": 1, ""metadata"": 2, ""primary_da...","{""csv"": 4, ""json"": 1, ""php"": 1, ""sql"": 1, ""unk...",4,6,"{""requested_strategy"": ""section_focused_contex...",donors | data/upload/donors.csv | donors.csv; ...,donors | data/upload/donors.csv | donors.csv; ...,donors | data/upload/donors.csv | donors.csv; ...
4,s41597-019-0342-9,/mydata/doc2validate/data/downloaded_artifacts...,has_real_artifact,has_extracted_tabular_like_candidate,github,,github,True,True,True,...,binary; text; unknown,text,"{""derived_data"": 1, ""metadata"": 1, ""primary_da...","{""binary"": 1, ""text"": 1, ""unknown"": 1}",2,5,"{""requested_strategy"": ""section_focused_contex...",pgxcorpus_annotations | PGxCorpus.tar | PGxCor...,pgxcorpus_annotations | PGxCorpus.tar | PGxCor...,


In [9]:
def decide_candidate(row):
    has_real=row.get("artifact_presence")=="has_real_artifact"
    has_extracted_strict_tabular=row.get("extracted_strict_tabular_file_count",0)>0
    has_extracted_tabular=row.get("extracted_tabular_like_file_count",0)>0
    has_schema=bool(row.get("has_dataset_structure")); has_primary_tabular=bool(row.get("has_primary_tabular")); has_column_semantics=bool(row.get("has_column_semantics")); has_known_path=row.get("known_path_or_pattern_count",0)>0
    non_tabular_or_env_count=row.get("non_tabular_or_env_logical_file_count",0); software_count=row.get("software_file_count",0)
    if not has_real: return "exclude_no_downloaded_content"
    if not has_schema: return "maybe_schema_missing_but_tabular_downloaded" if has_extracted_strict_tabular else "exclude_no_schema"
    if has_extracted_strict_tabular and has_column_semantics and has_known_path: return "yes_high_priority"
    if has_extracted_strict_tabular and has_primary_tabular and has_column_semantics: return "yes_after_path_manual_check"
    if has_extracted_strict_tabular and has_column_semantics: return "yes_after_role_or_path_manual_check"
    if has_extracted_tabular and has_column_semantics: return "yes_after_format_manual_check"
    if has_extracted_strict_tabular and has_schema: return "maybe_schema_or_column_semantics_missing"
    if non_tabular_or_env_count>0 or software_count>0: return "no_non_tabular_or_env_heavy"
    return "exclude_or_low_priority"

def decide_runtime_readiness(row):
    if row.get("artifact_presence") != "has_real_artifact": return "none_no_downloaded_content"
    if row.get("extracted_strict_tabular_file_count",0)>0:
        if row.get("non_tabular_or_env_logical_file_count",0)>0 or row.get("software_file_count",0)>0: return "medium_tabular_present_but_env_components"
        return "high_general_tabular_loader"
    if row.get("extracted_tabular_like_file_count",0)>0: return "medium_json_txt_or_non_strict_tabular"
    if row.get("non_tabular_or_env_logical_file_count",0)>0 or row.get("software_file_count",0)>0: return "low_specialized_runtime"
    return "unknown_or_low"

def candidate_reason(row):
    parts=[]
    if row.get("extracted_strict_tabular_file_count",0)>0: parts.append("extracted_strict_tabular_present")
    elif row.get("extracted_tabular_like_file_count",0)>0: parts.append("extracted_tabular_like_present")
    else: parts.append("no_extracted_tabular")
    parts.append("has_schema" if row.get("has_dataset_structure") else "missing_schema")
    parts.append("primary_tabular" if row.get("has_primary_tabular") else "no_primary_tabular")
    parts.append("has_column_semantics" if row.get("has_column_semantics") else "no_column_semantics")
    parts.append("has_path_or_pattern" if row.get("known_path_or_pattern_count",0)>0 else "no_path_or_pattern")
    if row.get("has_validation_targets"): parts.append("has_validation_targets")
    if row.get("has_execution_relevant_notes"): parts.append("has_execution_notes")
    if row.get("non_tabular_or_env_logical_file_count",0)>0: parts.append("has_non_tabular_or_env_logical_files")
    if row.get("software_file_count",0)>0: parts.append("has_software_files")
    return "; ".join(parts)

df["candidate_for_curated_benchmark_v1"] = df.apply(decide_candidate, axis=1)
df["runtime_readiness_v1"] = df.apply(decide_runtime_readiness, axis=1)
df["candidate_reason_v1"] = df.apply(candidate_reason, axis=1)

print("Total article dirs:", len(df))
print("Has real artifact:", (df["artifact_presence"]=="has_real_artifact").sum())
print("Has extracted content:", df["has_extracted_content"].sum())
print("Has extracted strict tabular:", (df["extracted_strict_tabular_file_count"]>0).sum())
print("Has dataset_structure:", df["has_dataset_structure"].sum())
print("Has column semantics:", df["has_column_semantics"].sum())
print("Has primary tabular:", df["has_primary_tabular"].sum())
print("Has path or pattern:", (df["known_path_or_pattern_count"]>0).sum())
display(df["candidate_for_curated_benchmark_v1"].value_counts(dropna=False).to_frame("count"))
display(df["runtime_readiness_v1"].value_counts(dropna=False).to_frame("count"))

Total article dirs: 187
Has real artifact: 113
Has extracted content: 103
Has extracted strict tabular: 72
Has dataset_structure: 187
Has column semantics: 180
Has primary tabular: 98
Has path or pattern: 185


,count
exclude_no_downloaded_content,74
yes_high_priority,70
no_non_tabular_or_env_heavy,22
yes_after_format_manual_check,12
exclude_or_low_priority,7
maybe_schema_or_column_semantics_missing,2


,count
none_no_downloaded_content,74
high_general_tabular_loader,41
medium_tabular_present_but_env_components,31
low_specialized_runtime,21
medium_json_txt_or_non_strict_tabular,13
unknown_or_low,7


In [10]:
df_high = df[df["candidate_for_curated_benchmark_v1"] == "yes_high_priority"].copy()
df_rerun_ready = df[(df["candidate_for_curated_benchmark_v1"] == "yes_high_priority") & (df["runtime_readiness_v1"] == "high_general_tabular_loader")].copy()
df_manual_check_pool = df[(df["candidate_for_curated_benchmark_v1"] == "yes_high_priority") & (df["runtime_readiness_v1"] == "medium_tabular_present_but_env_components")].copy()
expandable_labels=["yes_high_priority", "yes_after_path_manual_check", "yes_after_role_or_path_manual_check", "yes_after_format_manual_check", "maybe_schema_or_column_semantics_missing", "maybe_schema_missing_but_tabular_downloaded"]
df_likely_expandable = df[df["candidate_for_curated_benchmark_v1"].isin(expandable_labels)].copy()
df_downloaded_tabular = df[df["extracted_strict_tabular_file_count"] > 0].copy()
print("High priority:", len(df_high))
print("Rerun-ready v1:", len(df_rerun_ready))
print("Manual-check pool v1:", len(df_manual_check_pool))
print("Likely expandable:", len(df_likely_expandable))
print("Downloaded extracted strict tabular:", len(df_downloaded_tabular))

High priority: 70
Rerun-ready v1: 41
Manual-check pool v1: 29
Likely expandable: 84
Downloaded extracted strict tabular: 72


In [11]:
df.to_csv(OUTPUT_CSV, index=False)
df_no_content = df[df["artifact_presence"] != "has_real_artifact"].copy()
try:
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="all")
        df_high.to_excel(writer, index=False, sheet_name="high_priority")
        df_rerun_ready.to_excel(writer, index=False, sheet_name="rerun_ready")
        df_manual_check_pool.to_excel(writer, index=False, sheet_name="manual_check_pool")
        df_likely_expandable.to_excel(writer, index=False, sheet_name="likely_expandable")
        df_downloaded_tabular.to_excel(writer, index=False, sheet_name="downloaded_tabular")
        df_no_content.to_excel(writer, index=False, sheet_name="no_real_artifact")
        df["candidate_for_curated_benchmark_v1"].value_counts(dropna=False).to_frame("count").to_excel(writer, sheet_name="summary_candidate")
        df["runtime_readiness_v1"].value_counts(dropna=False).to_frame("count").to_excel(writer, sheet_name="summary_runtime")
        df["downloaded_pool_status_v2"].value_counts(dropna=False).to_frame("count").to_excel(writer, sheet_name="summary_download")
    print("Saved Excel:", OUTPUT_XLSX)
except Exception as e:
    print("Excel export failed:", repr(e))
print("Saved CSV:", OUTPUT_CSV)

Saved Excel: /mydata/doc2validate/results/runs/scidata_4293/analysis/downloaded_artifact_pool_audit_v3_3_schema.xlsx
Saved CSV: /mydata/doc2validate/results/runs/scidata_4293/analysis/downloaded_artifact_pool_audit_v3_3_schema.csv


## Downloader-aware URL worklist

This section directly reads validation JSON files and uses the exact URL selection rule from the downloader: `accessible is True`, then `redirected_url or url`.

In [12]:
def normalize_url_for_compare(url): return "" if not isinstance(url, str) else url.strip().rstrip("/")
def domain_of(url):
    try: return urlparse(normalize_url_for_compare(url)).netloc.lower().replace("www.", "")
    except Exception: return ""
def is_github(url): return "github.com" in domain_of(url)
def is_zenodo(url): return "zenodo.org" in domain_of(url)
def is_figshare(url): return "figshare" in domain_of(url)
def is_osf(url):
    d=domain_of(url); return d == "osf.io" or d.endswith(".osf.io")
def is_external_manual_platform(url): return is_zenodo(url) or is_figshare(url) or is_osf(url)
def dedup_keep_order(urls):
    seen=set(); out=[]
    for u in urls:
        nu=normalize_url_for_compare(u)
        if nu and nu not in seen:
            seen.add(nu); out.append(nu)
    return out

def get_validation_paths_like_downloader(article_id):
    validation_dir = DATA_ROOT / "structured_docs" / article_id / "validation"
    dataset_path = validation_dir / "dataset_url_validation.json"
    code_path = validation_dir / "code_repository_validation.json"
    return validation_dir, dataset_path, code_path

def collect_accessible_urls_from_validation_file(path: Path):
    if not Path(path).exists(): return []
    data=load_json(path)
    urls=[]
    for item in data.get("results", []):
        if item.get("accessible") is True:
            url=item.get("redirected_url") or item.get("url")
            if url: urls.append(url)
    return dedup_keep_order(urls)

def debug_validation_paths(article_id):
    validation_dir, dataset_path, code_path = get_validation_paths_like_downloader(article_id)
    print("article_id:", article_id)
    print("validation_dir:", validation_dir)
    print("dataset_path:", dataset_path, dataset_path.exists())
    print("code_path:", code_path, code_path.exists())
    if not dataset_path.exists():
        print("Searching DATA_ROOT for validation file...")
        for m in list(DATA_ROOT.rglob(f"{article_id}/validation/dataset_url_validation.json"))[:20]: print(m)
    if dataset_path.exists():
        data=load_json(dataset_path)
        print("dataset keys:", list(data.keys()))
        print("dataset results len:", len(data.get("results", [])))
        if data.get("results"): print(json.dumps(data["results"][0], ensure_ascii=False, indent=2)[:1500])

In [13]:
def extract_url_features_like_downloader(article_id):
    validation_dir, dataset_path, code_path = get_validation_paths_like_downloader(article_id)
    dataset_urls = collect_accessible_urls_from_validation_file(dataset_path)
    code_urls = collect_accessible_urls_from_validation_file(code_path)
    github_dataset_urls=[u for u in dataset_urls if is_github(u)]
    other_dataset_urls=[u for u in dataset_urls if not is_github(u)]
    external_manual_urls=[u for u in other_dataset_urls if is_external_manual_platform(u)]
    github_code_urls=[u for u in code_urls if is_github(u)]
    github_dataset_url=github_dataset_urls[0] if github_dataset_urls else ""
    code_repository_url=github_code_urls[0] if github_code_urls else (code_urls[0] if code_urls else "")
    code_equals_dataset=normalize_url_for_compare(github_dataset_url) != "" and normalize_url_for_compare(github_dataset_url) == normalize_url_for_compare(code_repository_url)
    manual_download_likely=len(external_manual_urls)>0
    if manual_download_likely and code_equals_dataset: priority_note="highest_priority: external official dataset URL exists, and GitHub dataset URL equals code repository URL"
    elif manual_download_likely: priority_note="manual_download_likely: Zenodo/Figshare/OSF URL exists"
    elif code_equals_dataset: priority_note="check_github_role: GitHub dataset URL equals code repository URL"
    elif github_dataset_url: priority_note="github_dataset_url_available"
    else: priority_note="no_accessible_github_dataset_url"
    return {
        "validation_dir": str(validation_dir), "dataset_validation_path": str(dataset_path), "code_repository_validation_path": str(code_path),
        "other_dataset_url": "; ".join(other_dataset_urls), "github_dataset_url": github_dataset_url, "code_repository_url": code_repository_url,
        "code_repository_equals_github_dataset_url": code_equals_dataset, "manual_download_likely": manual_download_likely, "external_manual_dataset_url": "; ".join(external_manual_urls),
        "all_accessible_dataset_urls": "; ".join(dataset_urls), "all_accessible_code_repository_urls": "; ".join(code_urls), "priority_note": priority_note,
    }

# Test one article before building worklists
test_id = df_high["article_id"].iloc[0] if len(df_high) else df["article_id"].iloc[0]
debug_validation_paths(test_id)
print(json.dumps(extract_url_features_like_downloader(test_id), ensure_ascii=False, indent=2))

article_id: s41597-019-0035-4
validation_dir: /mydata/doc2validate/data/structured_docs/s41597-019-0035-4/validation
dataset_path: /mydata/doc2validate/data/structured_docs/s41597-019-0035-4/validation/dataset_url_validation.json True
code_path: /mydata/doc2validate/data/structured_docs/s41597-019-0035-4/validation/code_repository_validation.json True
dataset keys: ['validator', 'article_id', 'status', 'validated_count', 'results']
dataset results len: 3
{
  "raw_url": "https://mriqc.nimh.nih.gov/",
  "redirected_url": "https://mriqc.nimh.nih.gov/",
  "accessible": true,
  "repo_type": "web",
  "estimated_size_bytes": null,
  "estimation_method": "not_estimated",
  "url_validity": {
    "is_valid": true,
    "reason": "ok"
  }
}
{
  "validation_dir": "/mydata/doc2validate/data/structured_docs/s41597-019-0035-4/validation",
  "dataset_validation_path": "/mydata/doc2validate/data/structured_docs/s41597-019-0035-4/validation/dataset_url_validation.json",
  "code_repository_validation_path

In [14]:
def build_url_worklist_like_downloader(subset_df, output_name):
    rows=[]
    for _, row in subset_df.iterrows():
        article_id=row["article_id"]
        url_feats=extract_url_features_like_downloader(article_id)
        rows.append({
            "article_id": article_id,
            "other_dataset_url": url_feats["other_dataset_url"],
            "github_dataset_url": url_feats["github_dataset_url"],
            "code_repository_url": url_feats["code_repository_url"],
            "code_repository_equals_github_dataset_url": url_feats["code_repository_equals_github_dataset_url"],
            "manual_download_likely": url_feats["manual_download_likely"],
            "external_manual_dataset_url": url_feats["external_manual_dataset_url"],
            "all_accessible_dataset_urls": url_feats["all_accessible_dataset_urls"],
            "all_accessible_code_repository_urls": url_feats["all_accessible_code_repository_urls"],
            "candidate_for_curated_benchmark_v1": row.get("candidate_for_curated_benchmark_v1", ""),
            "runtime_readiness_v1": row.get("runtime_readiness_v1", ""),
            "extracted_strict_tabular_file_count": row.get("extracted_strict_tabular_file_count", ""),
            "primary_tabular_file_count": row.get("primary_tabular_file_count", ""),
            "files_with_column_semantics_count": row.get("files_with_column_semantics_count", ""),
            "known_path_or_pattern_count": row.get("known_path_or_pattern_count", ""),
            "priority_note": url_feats["priority_note"],
            "validation_dir": url_feats["validation_dir"],
            "dataset_validation_path": url_feats["dataset_validation_path"],
            "code_repository_validation_path": url_feats["code_repository_validation_path"],
        })
    out=pd.DataFrame(rows)
    out=out.sort_values(by=["manual_download_likely", "code_repository_equals_github_dataset_url", "runtime_readiness_v1", "article_id"], ascending=[False, False, True, True])
    output_path=ANALYSIS_DIR / output_name
    out.to_csv(output_path, index=False)
    print("Saved:", output_path)
    print("Rows:", len(out))
    print("manual_download_likely:", int(out["manual_download_likely"].sum()) if len(out) else 0)
    print("code_repository_equals_github_dataset_url:", int(out["code_repository_equals_github_dataset_url"].sum()) if len(out) else 0)
    return out

df_rerun_ready_url_worklist = build_url_worklist_like_downloader(df_rerun_ready, "rerun_ready_v1_url_worklist.csv")
df_manual_check_url_worklist = build_url_worklist_like_downloader(df_manual_check_pool, "manual_check_pool_v1_url_worklist.csv")
df_high_priority_pool = pd.concat([df_rerun_ready.assign(pool_type="rerun_ready_v1"), df_manual_check_pool.assign(pool_type="manual_check_pool_v1")], ignore_index=True)
df_high_priority_url_worklist = build_url_worklist_like_downloader(df_high_priority_pool, "high_priority_70_url_worklist.csv")
pool_type_map = dict(zip(df_high_priority_pool["article_id"], df_high_priority_pool["pool_type"]))
df_high_priority_url_worklist["pool_type"] = df_high_priority_url_worklist["article_id"].map(pool_type_map)
cols = ["pool_type"] + [c for c in df_high_priority_url_worklist.columns if c != "pool_type"]
df_high_priority_url_worklist = df_high_priority_url_worklist[cols]
output_path = ANALYSIS_DIR / "high_priority_70_url_worklist.csv"
df_high_priority_url_worklist.to_csv(output_path, index=False)
print("Saved combined worklist:", output_path)
print("Rows:", len(df_high_priority_url_worklist))
display(df_high_priority_url_worklist.head(30))

Saved: /mydata/doc2validate/results/runs/scidata_4293/analysis/rerun_ready_v1_url_worklist.csv
Rows: 41
manual_download_likely: 19
code_repository_equals_github_dataset_url: 34
Saved: /mydata/doc2validate/results/runs/scidata_4293/analysis/manual_check_pool_v1_url_worklist.csv
Rows: 29
manual_download_likely: 7
code_repository_equals_github_dataset_url: 23
Saved: /mydata/doc2validate/results/runs/scidata_4293/analysis/high_priority_70_url_worklist.csv
Rows: 70
manual_download_likely: 26
code_repository_equals_github_dataset_url: 57
Saved combined worklist: /mydata/doc2validate/results/runs/scidata_4293/analysis/high_priority_70_url_worklist.csv
Rows: 70


,pool_type,article_id,other_dataset_url,github_dataset_url,code_repository_url,code_repository_equals_github_dataset_url,manual_download_likely,external_manual_dataset_url,all_accessible_dataset_urls,all_accessible_code_repository_urls,candidate_for_curated_benchmark_v1,runtime_readiness_v1,extracted_strict_tabular_file_count,primary_tabular_file_count,files_with_column_semantics_count,known_path_or_pattern_count,priority_note,validation_dir,dataset_validation_path,code_repository_validation_path
0,rerun_ready_v1,s41597-019-0035-4,https://mriqc.nimh.nih.gov; https://figshare.c...,https://github.com/oesteban/mriqc-webapi-snapshot,https://github.com/oesteban/mriqc-webapi-snapshot,True,True,https://figshare.com/articles/MRIQC_WebAPI_-_D...,https://mriqc.nimh.nih.gov; https://figshare.c...,https://github.com/oesteban/mriqc-webapi-snapshot,yes_high_priority,high_general_tabular_loader,7,1,6,6,highest_priority: external official dataset UR...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...
3,rerun_ready_v1,s41597-020-00688-8,https://figshare.com/articles/dataset/Our_Worl...,https://github.com/owid/covid-19-data/tree/mas...,https://github.com/owid/covid-19-data/tree/mas...,True,True,https://figshare.com/articles/dataset/Our_Worl...,https://github.com/owid/covid-19-data/tree/mas...,https://github.com/owid/covid-19-data/tree/mas...,yes_high_priority,high_general_tabular_loader,1613,2,6,6,highest_priority: external official dataset UR...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...
6,rerun_ready_v1,s41597-020-00788-5,https://springernature.figshare.com/collection...,https://github.com/MegaPast2Future/HerbiTraits,https://github.com/MegaPast2Future/HerbiTraits,True,True,https://springernature.figshare.com/collection...,https://springernature.figshare.com/collection...,https://github.com/MegaPast2Future/HerbiTraits,yes_high_priority,high_general_tabular_loader,15,2,6,7,highest_priority: external official dataset UR...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...
9,rerun_ready_v1,s41597-021-00955-2,https://opencovid.ca/api; https://springernatu...,https://github.com/ccodwg/Covid19Canada,https://github.com/ccodwg/Covid19Canada,True,True,https://springernature.figshare.com/articles/d...,https://github.com/ccodwg/Covid19Canada; https...,https://github.com/ccodwg/Covid19Canada; https...,yes_high_priority,high_general_tabular_loader,47,3,7,8,highest_priority: external official dataset UR...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...
11,rerun_ready_v1,s41597-021-01069-5,https://figshare.com; https://pypi.org/project...,https://github.com/theochem/B3DB,https://github.com/theochem/B3DB,True,True,https://figshare.com,https://github.com/theochem/B3DB; https://figs...,https://github.com/theochem/B3DB,yes_high_priority,high_general_tabular_loader,54,2,5,5,highest_priority: external official dataset UR...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...
13,rerun_ready_v1,s41597-022-01146-3,https://figshare.com/collections/Tropical_larv...,https://github.com/open-AIMS/tropical_ucrit_data,https://github.com/open-AIMS/tropical_ucrit_data,True,True,https://figshare.com/collections/Tropical_larv...,https://github.com/open-AIMS/tropical_ucrit_da...,https://github.com/open-AIMS/tropical_ucrit_data,yes_high_priority,high_general_tabular_loader,27,6,8,8,highest_priority: external official dataset UR...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...,/mydata/doc2validate/data/structured_docs/s415...
15

## If URL columns are empty

Run:

```python
debug_validation_paths("ARTICLE_ID")
```

If `dataset_path` is `False`, update `get_validation_paths_like_downloader()` to match the printed search result path.